## Встраивания

В нашем предыдущем примере мы работали с высокоразмерными векторами мешка слов длиной `vocab_size` и явно преобразовывали низкоразмерные векторы позиционного представления в разреженное one-hot представление. Это one-hot представление неэффективно с точки зрения памяти, кроме того, каждое слово рассматривается независимо друг от друга, то есть one-hot закодированные векторы не отражают семантическую близость между словами.

В этом разделе мы продолжим изучать набор данных **News AG**. Для начала загрузим данные и получим некоторые определения из предыдущего ноутбука.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Что такое embedding?

Идея **embedding** состоит в том, чтобы представить слова в виде плотных векторов меньшей размерности, которые каким-то образом отражают семантическое значение слова. Позже мы обсудим, как создавать осмысленные векторные представления слов, а пока давайте просто считать embedding способом уменьшения размерности вектора слова.

Итак, слой embedding будет принимать слово на вход и выдавать выходной вектор заданной длины `embedding_size`. В некотором смысле он очень похож на слой `Linear`, но вместо вектора с one-hot кодировкой сможет принимать номер слова на вход.

Используя слой embedding в качестве первого слоя нашей сети, мы можем перейти от модели bag-of-words к модели **embedding bag**, где сначала каждое слово в нашем тексте конвертируется в соответствующий embedding, а затем по всем этим embedding вычисляется агрегирующая функция, такая как `sum`, `average` или `max`.

![Изображение, показывающее классификатор с embedding для последовательности из пяти слов.](../../../../../translated_images/ru/embedding-classifier-example.b77f021a7ee67eee.webp)

Наша классифицирующая нейросеть начнётся со слоя embedding, затем пойдёт слой агрегации, и сверху — линейный классификатор:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Обработка переменного размера последовательности

В результате такой архитектуры минибатчи для нашей нейронной сети должны создаваться особым образом. В предыдущем разделе, при использовании мешка слов, все тензоры BoW в минибатче имели одинаковый размер `vocab_size`, независимо от фактической длины нашей текстовой последовательности. После перехода к векторным представлениям слов количество слов в каждом текстовом примере может варьироваться, и при объединении таких примеров в минибатчи необходимо применять паддинг.

Это можно сделать с помощью той же техники — предоставления функции `collate_fn` источнику данных:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Обучение классификатора на эмбеддингах

Теперь, когда мы определили подходящий загрузчик данных, мы можем обучить модель, используя функцию обучения, которую мы определили в предыдущем разделе:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Примечание**: Здесь мы обучаемся только на 25 тысячах записей (менее одного полного прохода) ради экономии времени, но вы можете продолжить обучение, написать функцию для обучения на нескольких эпохах и экспериментировать с параметром скорости обучения, чтобы достичь более высокой точности. Вы должны суметь достичь точности около 90%.


### Слой EmbeddingBag и представление последовательностей переменной длины

В предыдущей архитектуре нам приходилось дополнять все последовательности до одинаковой длины, чтобы поместить их в минибатч. Это не самый эффективный способ представления последовательностей переменной длины — другой подход заключается в использовании вектора **offset**, который будет содержать смещения всех последовательностей, хранящихся в одном большом векторе.

![Изображение, показывающее представление последовательности с помощью offset](../../../../../translated_images/ru/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Примечание**: На изображении выше показана последовательность символов, но в нашем примере мы работаем с последовательностями слов. Тем не менее общий принцип представления последовательностей с помощью вектора offset остаётся прежним.

Чтобы работать с представлением offset, мы используем слой [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html). Он похож на `Embedding`, но принимает на вход вектор содержимого и вектор offset, а также включает слой усреднения, который может быть `mean`, `sum` или `max`.

Вот модифицированная сеть, которая использует `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Для подготовки набора данных к обучению нам нужно предоставить функцию преобразования, которая подготовит вектор смещений:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Обратите внимание, что в отличие от всех предыдущих примеров, наша сеть теперь принимает два параметра: вектор данных и вектор смещений, которые имеют разные размеры. Аналогично, наш загрузчик данных теперь предоставляет нам 3 значения вместо 2: и текстовые, и векторные смещения используются в качестве признаков. Поэтому нам нужно немного скорректировать нашу функцию обучения, чтобы учесть это:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Семантические векторные представления: Word2Vec

В нашем предыдущем примере слой эмбеддингов модели научился отображать слова в векторные представления, однако это представление не имело большого семантического смысла. Было бы хорошо обучить такое векторное представление, при котором похожие слова или синонимы соответствовали бы векторам, которые находятся близко друг к другу с точки зрения некоторого векторного расстояния (например, евклидова расстояния).

Для этого нам нужно предварительно обучить нашу модель эмбеддингов на большом корпусе текстов особым образом. Один из первых способов обучить семантические векторные представления называется [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Он основан на двух основных архитектурах, которые используются для получения распределённого представления слов:

 - **Continuous bag-of-words** (CBoW) — в этой архитектуре мы обучаем модель предсказывать слово по окружающему контексту. Имея n-грамму $(W_{-2},W_{-1},W_0,W_1,W_2)$, целью модели является предсказать $W_0$ по $(W_{-2},W_{-1},W_1,W_2)$.
 - **Continuous skip-gram** — противоположность CBoW. Модель использует окно окружающего контекста, чтобы предсказать текущее слово.

CBoW работает быстрее, в то время как skip-gram медленнее, но лучше справляется с представлением редко встречающихся слов.

![Изображение, показывающее оба алгоритма CBoW и Skip-Gram для преобразования слов в векторы.](../../../../../translated_images/ru/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Чтобы поэкспериментировать с предобученными векторами word2vec, обученными на датасете Google News, мы можем использовать библиотеку **gensim**. Ниже мы найдём слова, наиболее похожие на 'neural'

> **Примечание:** При первом создании векторных представлений загрузка может занять некоторое время!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Мы также можем вычислить векторные эмбеддинги из слова для использования при обучении классификационной модели (для ясности показаны только первые 20 компонентов вектора):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Великое преимущество семантических встраиваний состоит в том, что вы можете манипулировать векторным кодированием для изменения семантики. Например, мы можем попросить найти слово, векторное представление которого будет максимально близко к словам *king* и *woman*, и максимально далеко от слова *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

И CBoW, и Skip-Grams являются «предсказательными» эмбеддингами, так как учитывают только локальные контексты. Word2Vec не использует глобальный контекст. 

**FastText** основывается на Word2Vec, обучая векторные представления для каждого слова и символьных n-грамм, найденных в каждом слове. Значения представлений затем усредняются в один вектор на каждом шаге обучения. Хотя это добавляет значительные вычислительные затраты при предобучении, оно позволяет эмбеддингам слов кодировать информацию о подсловах. 

Другой метод, **GloVe**, использует идею матрицы со-употребления, применяя нейронные методы для разложения матрицы со-употребления на более выразительные и нелинейные векторы слов.

Вы можете экспериментировать с примером, меняя эмбеддинги на FastText и GloVe, поскольку gensim поддерживает несколько различных моделей эмбеддингов слов.


## Использование предварительно обученных эмбеддингов в PyTorch

Мы можем изменить приведённый выше пример, чтобы предварительно заполнить матрицу в нашем слое эмбеддингов семантическими эмбеддингами, такими как Word2Vec. Необходимо учитывать, что словари предварительно обученного эмбеддинга и нашего текстового корпуса, скорее всего, не совпадут, поэтому мы инициализируем веса для отсутствующих слов случайными значениями:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Теперь давайте обучим нашу модель. Обратите внимание, что время, необходимое для обучения модели, значительно больше, чем в предыдущем примере, из-за большего размера слоя эмбеддингов и, следовательно, намного большего количества параметров. Также из-за этого нам, возможно, придется обучать нашу модель на большем количестве примеров, если мы хотим избежать переобучения.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

В нашем случае мы не видим значительного увеличения точности, что, вероятно, связано с очень разными словарями. 
Чтобы преодолеть проблему разных словарей, мы можем использовать одно из следующих решений:
* Переобучить модель word2vec на нашем словаре
* Загрузить наш набор данных с использованием словаря из предварительно обученной модели word2vec. Словарь, используемый для загрузки набора данных, можно указать во время загрузки.

Последний подход кажется проще, особенно потому что фреймворк PyTorch `torchtext` содержит встроенную поддержку для эмбеддингов. Мы можем, например, создать словарь на основе GloVe следующим образом:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Загруженный словарь имеет следующие базовые операции:
* `vocab.stoi` позволяет преобразовывать слово в его индекс в словаре
* `vocab.itos` выполняет обратное действие — преобразует число в слово
* `vocab.vectors` — это массив векторов эмбеддинга, поэтому чтобы получить эмбеддинг слова `s`, нам нужно использовать `vocab.vectors[vocab.stoi[s]]`

Вот пример работы с эмбеддингами, демонстрирующий уравнение **kind-man+woman = queen** (мне пришлось немного настроить коэффициент, чтобы это сработало):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Чтобы обучить классификатор с помощью этих встраиваний, нам сначала нужно закодировать наш набор данных, используя словарь GloVe:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Как мы уже видели выше, все векторные эмбеддинги хранятся в матрице `vocab.vectors`. Это очень облегчает загрузку этих весов в веса слоя эмбеддингов простым копированием:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Теперь давайте обучим нашу модель и посмотрим, получим ли мы лучшие результаты:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Одна из причин, по которой мы не наблюдаем значительного увеличения точности, заключается в том, что некоторые слова из нашего набора данных отсутствуют в предварительно обученном словаре GloVe, и, следовательно, они фактически игнорируются. Чтобы преодолеть этот факт, мы можем обучить собственные эмбеддинги на нашем наборе данных.


## Контекстные эмбеддинги

Одним из ключевых ограничений традиционных предобученных представлений эмбеддингов, таких как Word2Vec, является проблема дисамбигуации смыслов слов. Хотя предобученные эмбеддинги могут захватывать некоторое значение слов в контексте, каждое возможное значение слова кодируется в одном и том же эмбеддинге. Это может вызывать проблемы в последующих моделях, поскольку многие слова, такие как слово «play», имеют разные значения в зависимости от контекста их использования.

Например, слово «play» в этих двух разных предложениях имеет совершенно разное значение:
- Я ходил на **пьесу** в театр.
- Джон хочет **поиграть** с друзьями.

Предобученные эмбеддинги выше представляют оба этих значения слова «play» в одном и том же эмбеддинге. Чтобы преодолеть это ограничение, нам нужно строить эмбеддинги на основе **языковой модели**, которая обучена на большом корпусе текстов и *знает*, как слова могут сочетаться в разных контекстах. Обсуждение контекстных эмбеддингов выходит за рамки этого руководства, но мы вернемся к ним при обсуждении языковых моделей в следующем разделе.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
